# Step 2: MobileNetV4ConvSmall Training, Export, and Inference

This notebook trains the SoundPulse Tiny CNN replacement using TensorFlow's official `MobileNetV4ConvSmall` backbone. The notebook keeps data loading, feature generation, model definition, training, prototype extraction, TFLite export, and sample inference in one place.

In [1]:
#%pip install -q -e ..

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

PROJECT_ROOT = Path.cwd().resolve().parents[1]
ML_ROOT = PROJECT_ROOT / 'ml'
SRC_ROOT = ML_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from soundpulse_ml.datasets import load_dataset_manifest, manifest_to_dataframe

CONTRACTS_DIR = PROJECT_ROOT / 'packages' / 'shared_contracts' / 'v1'
CURATED_MANIFEST_PATH = ML_ROOT / 'data' / 'curated' / 'soundpulse_manifest.v1.json'
MODEL_OUTPUT_DIR = ML_ROOT / 'export' / 'artifacts' / 'generated' / 'tinycnn_baseline'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with (CONTRACTS_DIR / 'class_labels.json').open('r', encoding='utf-8') as handle:
    CLASS_CONTRACT = json.load(handle)
with (CONTRACTS_DIR / 'preprocessing_config.json').open('r', encoding='utf-8') as handle:
    PREPROCESSING_CONTRACT = json.load(handle)

CLASS_ORDER = [item['slug'] for item in CLASS_CONTRACT['classes']]
CLASS_TO_INDEX = {slug: index for index, slug in enumerate(CLASS_ORDER)}
SAMPLE_RATE = PREPROCESSING_CONTRACT['audio']['sample_rate_hz']
WINDOW_DURATION_MS = PREPROCESSING_CONTRACT['audio']['window_duration_ms']
WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_DURATION_MS / 1000)
IMAGE_SIZE = (128, 128)
EMBEDDING_DIM = PREPROCESSING_CONTRACT['embedding_contract']['tiny_cnn_dim']
BATCH_SIZE = 16
EPOCHS = 50
AUTOTUNE = tf.data.AUTOTUNE
SHARED_TEST_CAP = 64
MAX_EXAMPLES_PER_CLASS_BY_SPLIT = {'train': 256, 'validation': 96, 'test': SHARED_TEST_CAP}

tf.random.set_seed(42)
np.random.seed(42)

assert CURATED_MANIFEST_PATH.exists(), 'Run Notebook 1 first to build the curated manifest.'

print({
    'manifest_path': str(CURATED_MANIFEST_PATH),
    'class_order': CLASS_ORDER,
    'epochs': EPOCHS,
    'max_examples_per_class_by_split': MAX_EXAMPLES_PER_CLASS_BY_SPLIT,
})

{'manifest_path': 'C:\\Users\\ZBOOK STUDIO G5\\Documents\\GitHub\\SoundPulse\\ml\\data\\curated\\soundpulse_manifest.v1.json', 'class_order': ['fire_alarm', 'clock_alarm', 'door_knock', 'doorbell', 'crying_baby', 'water_running', 'dog_barking', 'door_wood_creaking', 'speech', 'background'], 'epochs': 50, 'max_examples_per_class_by_split': {'train': 256, 'validation': 96, 'test': 64}}


In [3]:
manifest = load_dataset_manifest(CURATED_MANIFEST_PATH)
manifest_df = manifest_to_dataframe(manifest)
manifest_df = manifest_df[manifest_df['audio_path'].map(lambda value: Path(value).exists())].reset_index(drop=True)

def cap_examples_per_class(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    split_frame = frame[frame['split'] == split_name].copy()
    max_examples = MAX_EXAMPLES_PER_CLASS_BY_SPLIT.get(split_name)
    if max_examples is None:
        return split_frame.reset_index(drop=True)
    return (
        split_frame
        .sort_values(['class_slug', 'clip_id'])
        .groupby('class_slug', group_keys=False)
        .head(max_examples)
        .reset_index(drop=True)
    )

manifest_df = pd.concat(
    [cap_examples_per_class(manifest_df, split_name) for split_name in ['train', 'validation', 'test']],
    ignore_index=True,
)

manifest_df.groupby(['split', 'class_slug']).size().unstack(fill_value=0)

class_slug,clock_alarm,crying_baby,dog_barking,door_knock,door_wood_creaking,doorbell,fire_alarm,speech,water_running
split,,,,,,,,,
test,8,38,64,64,64,34,64,64,64
train,24,114,256,248,256,90,256,256,237
validation,8,25,69,50,62,17,74,96,40


In [4]:
def load_audio(audio_path: tf.Tensor) -> tf.Tensor:
    path = audio_path.numpy().decode('utf-8')
    waveform, sample_rate = sf.read(path, dtype='float32', always_2d=False)
    if waveform.ndim > 1:
        waveform = np.mean(waveform, axis=1)
    if sample_rate != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=SAMPLE_RATE)
    if waveform.shape[0] < WINDOW_SAMPLES:
        waveform = np.pad(waveform, (0, WINDOW_SAMPLES - waveform.shape[0]))
    else:
        waveform = waveform[:WINDOW_SAMPLES]
    return waveform.astype(np.float32)


def waveform_to_rgb_log_mel(waveform: tf.Tensor) -> tf.Tensor:
    stft = tf.signal.stft(
        waveform,
        frame_length=PREPROCESSING_CONTRACT['stft']['win_length'],
        frame_step=PREPROCESSING_CONTRACT['stft']['hop_length'],
        fft_length=PREPROCESSING_CONTRACT['stft']['n_fft'],
        window_fn=tf.signal.hann_window,
        pad_end=False,
    )
    spectrogram = tf.abs(stft) ** PREPROCESSING_CONTRACT['mel']['power']
    mel_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=PREPROCESSING_CONTRACT['mel']['n_mels'],
        num_spectrogram_bins=spectrogram.shape[-1],
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=PREPROCESSING_CONTRACT['mel']['f_min_hz'],
        upper_edge_hertz=PREPROCESSING_CONTRACT['mel']['f_max_hz'],
    )
    mel_spectrogram = tf.matmul(spectrogram, mel_matrix)
    log_mel = tf.math.log(mel_spectrogram + PREPROCESSING_CONTRACT['mel']['log_offset'])
    log_mel = tf.clip_by_value(
        log_mel,
        PREPROCESSING_CONTRACT['normalization']['clip_min_db'],
        PREPROCESSING_CONTRACT['normalization']['clip_max_db'],
    )
    log_mel = tf.expand_dims(log_mel, axis=-1)
    log_mel = tf.image.resize(log_mel, IMAGE_SIZE)
    log_mel = tf.repeat(log_mel, repeats=3, axis=-1)
    log_mel = tf.image.per_image_standardization(log_mel)
    return log_mel


def load_example(audio_path: tf.Tensor, label: tf.Tensor):
    waveform = tf.py_function(load_audio, [audio_path], Tout=tf.float32)
    waveform.set_shape([WINDOW_SAMPLES])
    image = waveform_to_rgb_log_mel(waveform)
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return image, label

In [5]:
def make_dataset(split_name: str, training: bool) -> tf.data.Dataset:
    frame = manifest_df[manifest_df['split'] == split_name].copy()
    assert not frame.empty, f'No examples available for split: {split_name}'
    labels = frame['class_slug'].map(CLASS_TO_INDEX).astype(np.int32).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((frame['audio_path'].to_numpy(), labels))
    if training:
        dataset = dataset.shuffle(buffer_size=len(frame), seed=42, reshuffle_each_iteration=True)
    dataset = dataset.map(load_example, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset

train_ds = make_dataset('train', training=True)
validation_ds = make_dataset('validation', training=False)
test_ds = make_dataset('test', training=False)

train_ds

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [6]:
def build_tiny_cnn_classifier(num_classes: int) -> tuple[tf.keras.Model, tf.keras.Model]:
    inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name='log_mel_rgb')
    x = tf.keras.layers.Conv2D(32, kernel_size=3, padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)
    x = tf.keras.layers.Conv2D(64, kernel_size=3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)
    x = tf.keras.layers.Conv2D(128, kernel_size=3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    embedding = tf.keras.layers.Dense(EMBEDDING_DIM, name='embedding')(x)
    normalized_embedding = tf.keras.layers.Lambda(
        lambda tensor: tf.math.l2_normalize(tensor, axis=-1),
        name='embedding_norm',
    )(embedding)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='classifier')(normalized_embedding)
    classifier = tf.keras.Model(inputs=inputs, outputs=outputs, name='soundpulse_tinycnn_classifier')
    embedding_model = tf.keras.Model(inputs=inputs, outputs=normalized_embedding, name='soundpulse_tinycnn_embedding_model')
    return classifier, embedding_model

classifier, embedding_model = build_tiny_cnn_classifier(len(CLASS_ORDER))
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
classifier.summary()

Model: "soundpulse_tinycnn_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ log_mel_rgb (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_avg_pool                 │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Dense)               │ (None, 1024)           │       132,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_norm (Lambda)         │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 236,490 (923.79 KB)

 Trainable params: 236,042 (922.04 KB)

 Non-trainable params: 448 (1.75 KB)

In [7]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_OUTPUT_DIR / 'best.weights.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
]

history = classifier.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

Epoch 1/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 27s 204ms/step - accuracy: 0.4266 - loss: 1.8378 - val_accuracy: 0.1610 - val_loss: 2.3716
Epoch 2/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 23s 205ms/step - accuracy: 0.5037 - loss: 1.5072 - val_accuracy: 0.1224 - val_loss: 2.4471
Epoch 3/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 22s 204ms/step - accuracy: 0.5302 - loss: 1.3947 - val_accuracy: 0.2109 - val_loss: 2.6754
Epoch 4/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 23s 211ms/step - accuracy: 0.5527 - loss: 1.3397 - val_accuracy: 0.2630 - val_loss: 2.3406
Epoch 5/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 22s 205ms/step - accuracy: 0.5740 - loss: 1.2575 - val_accuracy: 0.2766 - val_loss: 2.4258
Epoch 6/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 22s 200ms/step - accuracy: 0.5734 - loss: 1.2032 - val_accuracy: 0.4036 - val_loss: 1.7198
Epoch 7/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 22s 201ms/step - accuracy: 0.5982 - loss: 1.1544 - val_accuracy: 0.3492 - val_loss: 1.9616
Epoch 8/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 23s 206ms/step - accuracy: 0.6166 - loss: 1

In [8]:
test_probabilities = classifier.predict(test_ds)
test_predictions = np.argmax(test_probabilities, axis=1)
test_labels = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)

print(
    classification_report(
        test_labels,
        test_predictions,
        labels=list(range(len(CLASS_ORDER))),
        target_names=CLASS_ORDER,
        zero_division=0,
    )
)
confusion_matrix(test_labels, test_predictions, labels=list(range(len(CLASS_ORDER))))

29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step
                    precision    recall  f1-score   support

        fire_alarm       0.25      0.52      0.34        64
       clock_alarm       0.00      0.00      0.00         8
        door_knock       0.66      0.62      0.64        64
          doorbell       0.33      0.03      0.05        34
       crying_baby       0.80      0.21      0.33        38
     water_running       0.39      0.42      0.40        64
       dog_barking       0.59      0.30      0.40        64
door_wood_creaking       0.22      0.19      0.20        64
            speech       0.25      0.39      0.30        64
        background       0.00      0.00      0.00         0

          accuracy                           0.36       464
         macro avg       0.35      0.27      0.27       464
      weighted avg       0.41      0.36      0.35       464



array([[33,  0,  3,  1,  0, 11,  0,  4, 12,  0],
       [ 3,  0,  0,  1,  0,  1,  0,  2,  1,  0],
       [11,  0, 40,  0,  0,  5,  0,  0,  8,  0],
       [17,  0,  1,  1,  1,  2,  1,  4,  7,  0],
       [ 6,  0,  0,  0,  8,  2,  4,  6, 12,  0],
       [12,  0,  4,  0,  0, 27,  0, 14,  7,  0],
       [ 9,  0,  3,  0,  0, 10, 19,  6, 17,  0],
       [21,  0,  9,  0,  1,  9,  0, 12, 12,  0],
       [21,  0,  1,  0,  0,  3,  8,  6, 25,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]])

In [9]:
def collect_embeddings(dataset: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray]:
    images = []
    labels = []
    for batch_images, batch_labels in dataset:
        images.append(embedding_model.predict(batch_images, verbose=0))
        labels.append(batch_labels.numpy())
    return np.concatenate(images, axis=0), np.concatenate(labels, axis=0)


train_embeddings, train_labels = collect_embeddings(train_ds)
prototype_store = {}
for class_index, class_slug in enumerate(CLASS_ORDER):
    mask = train_labels == class_index
    if mask.any():
        prototype_store[class_slug] = train_embeddings[mask].mean(axis=0).tolist()

with (MODEL_OUTPUT_DIR / 'prototype_store.json').open('w', encoding='utf-8') as handle:
    json.dump(prototype_store, handle, indent=2)

print(f'Prototype classes exported: {sorted(prototype_store.keys())}')

Prototype classes exported: ['clock_alarm', 'crying_baby', 'dog_barking', 'door_knock', 'door_wood_creaking', 'doorbell', 'fire_alarm', 'speech', 'water_running']


In [10]:
embedding_model.export(str(MODEL_OUTPUT_DIR / 'saved_model'))
converter = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_OUTPUT_DIR / 'saved_model'))
tflite_model = converter.convert()
with (MODEL_OUTPUT_DIR / 'tinycnn_embedding.tflite').open('wb') as handle:
    handle.write(tflite_model)

metadata = {
    'model_name': 'TinyCNN',
    'embedding_dim': EMBEDDING_DIM,
    'class_order': CLASS_ORDER,
    'image_size': IMAGE_SIZE,
    'sample_rate_hz': SAMPLE_RATE,
}
with (MODEL_OUTPUT_DIR / 'export_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'Exported TFLite embedding model to {MODEL_OUTPUT_DIR}')

INFO:tensorflow:Assets written to: C:\Users\ZBOOK STUDIO G5\Documents\GitHub\SoundPulse\ml\export\artifacts\generated\tinycnn_baseline\saved_model\assets


INFO:tensorflow:Assets written to: C:\Users\ZBOOK STUDIO G5\Documents\GitHub\SoundPulse\ml\export\artifacts\generated\tinycnn_baseline\saved_model\assets


Saved artifact at 'C:\Users\ZBOOK STUDIO G5\Documents\GitHub\SoundPulse\ml\export\artifacts\generated\tinycnn_baseline\saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='log_mel_rgb')
Output Type:
  TensorSpec(shape=(None, 1024), dtype=tf.float32, name=None)
Captures:
  2043805384528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2043805384144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2043821443024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2043821442640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2043821441296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2043821441872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2043821441680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2043821441104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2043821443984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  20438

In [16]:
sample_row = manifest_df.iloc[0]
sample_waveform = load_audio(tf.constant(sample_row['audio_path']))
sample_image = waveform_to_rgb_log_mel(tf.convert_to_tensor(sample_waveform))
sample_embedding = embedding_model.predict(tf.expand_dims(sample_image, axis=0), verbose=0)[0]

distances = {
    class_slug: float(np.sum((sample_embedding - np.array(prototype)) ** 2))
    for class_slug, prototype in prototype_store.items()
}
predicted_label = min(distances, key=distances.get)
print({'clip_id': sample_row['clip_id'], 'true_label': sample_row['class_slug'], 'predicted_label': predicted_label, 'distance': distances[predicted_label]})

{'clip_id': 'esc50-2-102435-A-37', 'true_label': 'clock_alarm', 'predicted_label': 'clock_alarm', 'distance': 0.2461881149394275}
